In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.join(os.getcwd(), 'config'))

from db_config import get_engine

In [2]:
engine = get_engine()
conn = engine.connect()
print("Connexion OK")

Connexion OK


In [3]:
dim_title   = pd.read_sql("SELECT * FROM silver.dim_title", conn)
dim_genre   = pd.read_sql("SELECT * FROM silver.dim_genre", conn)
dim_country = pd.read_sql("SELECT * FROM silver.dim_country", conn)
dim_cast    = pd.read_sql("SELECT * FROM silver.dim_cast", conn)

In [ ]:
# gold.dim_date
dim_date = dim_title[['date_added', 'added_year', 'added_month']].dropna().drop_duplicates()

dim_date = dim_date.copy()
dim_date['date_key']   = dim_date['date_added'].dt.strftime('%Y%m%d').astype(int)
dim_date['quarter']    = dim_date['date_added'].dt.quarter
dim_date['month_name'] = dim_date['date_added'].dt.strftime('%B')
dim_date['day']        = dim_date['date_added'].dt.day
dim_date['weekday']    = dim_date['date_added'].dt.strftime('%A')

dim_date.to_sql('dim_date', engine, schema='gold',
                if_exists='replace', index=False)
print(f" dim_date — {len(dim_date)} lignes")

 dim_date — 1714 lignes


In [5]:
# gold.dim_type
dim_type = dim_title[['type']].drop_duplicates().reset_index(drop=True)
dim_type['type_id'] = dim_type.index + 1

dim_type.to_sql('dim_type', engine, schema='gold',
                if_exists='replace', index=False)
print(f" dim_type - {len(dim_type)} lignes")
dim_type

 dim_type - 2 lignes


,type,type_id
0,Movie,1
1,TV Show,2


In [6]:
# gold.dim_rating
dim_rating = dim_title[['rating']].drop_duplicates().dropna().reset_index(drop=True)
dim_rating['rating_id'] = dim_rating.index + 1

# Ajouter description du rating
rating_desc = {
    'TV-MA': 'Mature Audiences',
    'TV-14': 'Parents Strongly Cautioned',
    'TV-PG': 'Parental Guidance',
    'TV-G' : 'General Audience',
    'TV-Y' : 'All Children',
    'TV-Y7': 'Children 7+',
    'PG-13': 'Parents Strongly Cautioned',
    'PG'   : 'Parental Guidance',
    'R'    : 'Restricted',
    'G'    : 'General Audience',
    'NC-17': 'Adults Only',
    'NR'   : 'Not Rated',
    'UR'   : 'Unrated'
}
dim_rating['rating_description'] = dim_rating['rating'].map(rating_desc).fillna('Unknown')

dim_rating.to_sql('dim_rating', engine, schema='gold',
                  if_exists='replace', index=False)
print(f" dim_rating — {len(dim_rating)} lignes")
dim_rating

 dim_rating — 17 lignes


,rating,rating_id,rating_description
0,PG-13,1,Parents Strongly Cautioned
1,TV-MA,2,Mature Audiences
2,PG,3,Parental Guidance
3,TV-14,4,Parents Strongly Cautioned
4,TV-PG,5,Parental Guidance
5,TV-Y,6,All Children
6,TV-Y7,7,Children 7+
7,R,8,Restricted
8,TV-G,9,General Audience
9,G,10,General Audience


In [7]:
# gold.dim_genre
dim_genre_gold = dim_genre[['genre']].drop_duplicates().dropna().reset_index(drop=True)
dim_genre_gold['genre_id'] = dim_genre_gold.index + 1

dim_genre_gold.to_sql('dim_genre', engine, schema='gold',
                      if_exists='replace', index=False)
print(f" dim_genre — {len(dim_genre_gold)} lignes")
dim_genre_gold

 dim_genre — 42 lignes


,genre,genre_id
0,Documentaries,1
1,International TV Shows,2
2,TV Dramas,3
3,TV Mysteries,4
4,Crime TV Shows,5
5,TV Action & Adventure,6
6,Docuseries,7
7,Reality TV,8
8,Romantic TV Shows,9
9,TV Comedies,10


In [8]:
#  gold.dim_country
dim_country_gold = dim_country[['country']].drop_duplicates().dropna().reset_index(drop=True)
dim_country_gold['country_id'] = dim_country_gold.index + 1

dim_country_gold.to_sql('dim_country', engine, schema='gold',
                        if_exists='replace', index=False)
print(f" dim_country — {len(dim_country_gold)} lignes")

 dim_country — 123 lignes


In [9]:
# gold.fact_netflix
fact = dim_title[[
    'show_id', 'type', 'title', 'director',
    'date_added', 'release_year', 'rating',
    'duration_int', 'duration_unit', 'description'
]].copy()

# Joindre les cles étrangeres
fact = fact.merge(dim_type,       on='type',   how='left')
fact = fact.merge(dim_rating,     on='rating', how='left')

# date_key
fact['date_key'] = pd.to_datetime(fact['date_added']).dt.strftime('%Y%m%d')
fact['date_key'] = pd.to_numeric(fact['date_key'], errors='coerce')

# Garder uniquement les colonnes utiles
fact_final = fact[[
    'show_id',
    'title',
    'director',
    'type_id',
    'rating_id',
    'date_key',
    'release_year',
    'duration_int',
    'duration_unit',
    'description'
]].copy()

fact_final.to_sql('fact_netflix', engine, schema='gold',
                  if_exists='replace', index=False)
print(f" fact_netflix — {len(fact_final)} lignes")

 fact_netflix — 8807 lignes


In [10]:
verif = pd.read_sql("""
    select
        (select COUNT(*) FROM gold.fact_netflix)  AS fact_netflix,
        (select COUNT(*) FROM gold.dim_date)      AS dim_date,
        (select COUNT(*) FROM gold.dim_type)      AS dim_type,
        (select COUNT(*) FROM gold.dim_rating)    AS dim_rating,
        (select COUNT(*) FROM gold.dim_genre)     AS dim_genre,
        (select COUNT(*) FROM gold.dim_country)   AS dim_country
""", engine)

print(verif)

   fact_netflix  dim_date  dim_type  dim_rating  dim_genre  dim_country
0          8807      1714         2          17         42          123
